# Chapter 10 — Financial Correlation Modeling — Bottom-Up Approaches

教材 pp. 130–133；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from copula_calculator import *

教材表10.1与式(10.6)，pp.132–133：一年联合违约概率。

In [2]:
correlation=pd.DataFrame([[1,.4],[.4,1]],index=['B','Caa'],columns=['B','Caa'])
copula=GaussianCopula(correlation)
probabilities=pd.DataFrame({'pd':[.0651,.2383]})
assert_allclose(copula.joint_probability(probabilities,'pd'),.0344,atol=.0001)
independent=GaussianCopula(pd.DataFrame(np.eye(2)))
assert_allclose(independent.joint_probability(probabilities,'pd'),.0651*.2383)
print(copula.joint_probability(probabilities,'pd'))

0.034427273818630755


模拟复现相关结构，默认时点逆变换可回代；p.133。

In [3]:
u=copula.sample(20000,np.random.default_rng(42))
z=pd.DataFrame(norm.ppf(u))
assert_allclose(z.corr().iloc[0,1],.4,atol=.025)
hazards=pd.DataFrame({'h':[.02,.05]})
t=exponential_default_times(u,['B','Caa'],hazards,'h')
assert_allclose(1-np.exp(-t.to_numpy()*np.array([.02,.05])),u)

## 非正态边际映射；pp.130–133

In [4]:
from scipy.stats import expon
mapped=map_marginals(u,{'B':expon.ppf,'Caa':norm.ppf})
assert_allclose(expon.cdf(mapped.B),u.B)
assert_allclose(norm.cdf(mapped.Caa),u.Caa)